# Tahap 2A: Transformasi Varians & Uji Stasioneritas Awal
**Penanggung Jawab:** Maulana Naufal H  
**Mata Kuliah:** Analisis Data Timeseries — Sains Data  

---

### Tujuan Pengerjaan:
1. **Evaluasi Kestabilan Varians:** Memeriksa hubungan mean-varians, menguji parameter $\lambda$ Box-Cox, dan membandingkan skala asli vs logaritma.
2. **Uji Stasioneritas Awal (Level):** Menjalankan uji ADF dan KPSS pada deret level (skala asli dan skala log) sebagai dasar penentuan orde diferensiasi ($d$ dan $D$) untuk Tahap 2B (Aldo).


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import boxcox, boxcox_llf
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore")

# Path data
ROOT = Path.cwd().resolve()
if (ROOT / "data" / "processed").exists():
    DATA_DIR = ROOT / "data" / "processed"
elif Path(r"C:\Users\Lenovo\Downloads\Analisis Data Time Series\data\processed").exists():
    DATA_DIR = Path(r"C:\Users\Lenovo\Downloads\Analisis Data Time Series\data\processed")
else:
    DATA_DIR = Path(r"C:\Users\Lenovo\.gemini\antigravity\scratch\adt-project\data\processed")

# Visualisasi
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "figure.autolayout": True
})

# Load dataset mingguan PM2.5 Surakarta (hasil pra-pemrosesan Kunto)
df = pd.read_csv(DATA_DIR / "pm25_surakarta_weekly_model.csv", parse_dates=["week_start"])
series = df.set_index("week_start")["pm25"].asfreq("7D")

# Partisi: 104 minggu train (2 siklus musiman tahunan) dan 12 minggu test
train = series.iloc[:-12]
test = series.iloc[-12:]

print(f"Total deret : {len(series)} minggu ({series.index.min().strftime('%Y-%m-%d')} s.d. {series.index.max().strftime('%Y-%m-%d')})")
print(f"Data latih  : {len(train)} minggu ({train.index.min().strftime('%Y-%m-%d')} s.d. {train.index.max().strftime('%Y-%m-%d')})")
print(f"Data uji    : {len(test)} minggu ({test.index.min().strftime('%Y-%m-%d')} s.d. {test.index.max().strftime('%Y-%m-%d')})")


## 1. Diagnosis Kestabilan Varians
Sebelum melakukan diferensiasi rataan, kestabilan varians diuji terlebih dahulu untuk melihat apakah variabilitas data membesar seiring naiknya level rata-rata (heteroskedastisitas).
* **Uji visual:** Scatter plot antara *rolling mean* dan *rolling standard deviation* (window = 8 minggu).
* **Uji segmen:** Perbandingan varians antarsegmen semester (26 minggu).


In [ ]:
roll_mean = train.rolling(8, center=True).mean().dropna()
roll_std = train.rolling(8, center=True).std().dropna()
corr, p_corr = stats.pearsonr(roll_mean, roll_std)

# Varians per semester (26 minggu)
sem_stats = []
for i in range(4):
    seg = train.iloc[i * 26:(i + 1) * 26]
    sem_stats.append({
        "Semester": f"Sem {i+1} ({seg.index.min().strftime('%b %y')} - {seg.index.max().strftime('%b %y')})",
        "Mean": seg.mean(),
        "Std": seg.std(),
        "Varians": seg.var(),
        "CV": seg.std() / seg.mean()
    })
df_sem = pd.DataFrame(sem_stats)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(roll_mean, roll_std, color="#1d4ed8", alpha=0.7)
m, b = np.polyfit(roll_mean, roll_std, 1)
ax[0].plot(roll_mean, m * roll_mean + b, color="#dc2626", linestyle="--", lw=1.2)
ax[0].set_title(f"Rolling Mean vs Std (r = {corr:.3f}, p = {p_corr:.3f})")
ax[0].set_xlabel("Mean PM2.5")
ax[0].set_ylabel("Std PM2.5")

ax[1].bar([f"Sem {i+1}" for i in range(4)], df_sem["Varians"], color="#0891b2", alpha=0.75)
ax[1].set_title("Varians Tiap Semester (Data Latih)")
ax[1].set_ylabel("Varians ((µg/m³)²) ")
plt.tight_layout()
plt.show()

display(df_sem.round(2))


## 2. Uji Parameter Box-Cox & Perbandingan Skala
Transformasi Box-Cox didefinisikan sebagai:
$$w_t = \begin{cases} \frac{y_t^\lambda - 1}{\lambda}, & \lambda \neq 0 \\[6pt] \ln(y_t), & \lambda = 0 \end{cases}$$

Untuk menguji apakah $\lambda$ berbeda secara signifikan dari 1 (deret asli) atau 0 (logaritma), digunakan uji Likelihood Ratio (LR):
$$LR = 2 [\ell(\hat{\lambda}) - \ell(\lambda_0)] \sim \chi^2_{(1)}$$


In [ ]:
train_bc, lam_opt = boxcox(train)
train_log = np.log(train)

# Likelihood Ratio Test
llf_opt = boxcox_llf(lam_opt, train)
llf_1 = boxcox_llf(1.0, train)
llf_0 = boxcox_llf(0.0, train)

lr_1 = 2 * (llf_opt - llf_1)
p_1 = 1 - stats.chi2.cdf(lr_1, df=1)

lr_0 = 2 * (llf_opt - llf_0)
p_0 = 1 - stats.chi2.cdf(lr_0, df=1)

df_lr = pd.DataFrame({
    "Hipotesis (H0)": ["λ = 1 (Deret Asli)", "λ = 0 (Transformasi Log)", "λ optimal (MLE)"],
    "Nilai λ": [1.0, 0.0, round(lam_opt, 4)],
    "Log-Likelihood": [round(llf_1, 2), round(llf_0, 2), round(llf_opt, 2)],
    "LR Stat": [round(lr_1, 3), round(lr_0, 3), 0.0],
    "p-value": [f"{p_1:.4e}", f"{p_0:.4f}", "-"],
    "Kesimpulan Statistik": [
        "Tolak H0 (λ=1 tidak didukung)" if p_1 < 0.05 else "Gagal tolak H0",
        "Gagal tolak H0 (λ=0 didukung)" if p_0 >= 0.05 else "Tolak H0",
        "Nilai Optimal"
    ]
})
print("Uji Hipotesis Parameter Lambda:")
display(df_lr)

# Plot perbandingan bentuk deret
fig, ax = plt.subplots(3, 1, figsize=(10, 6.5), sharex=True)
ax[0].plot(train.index, train.values, color="#1d4ed8")
ax[0].set_title("Deret Asli (µg/m³)")

ax[1].plot(train.index, train_log.values, color="#059669")
ax[1].set_title("Transformasi Logaritma Natural (λ = 0)")

ax[2].plot(train.index, train_bc, color="#d97706")
ax[2].set_title(f"Transformasi Box-Cox Optimal (λ = {lam_opt:.2f})")
plt.tight_layout()
plt.show()

# Rasio varians max/min semester
def var_ratio(arr):
    v = [arr[i * 26:(i + 1) * 26].var() for i in range(4)]
    return max(v) / min(v)

df_comp = pd.DataFrame({
    "Skala": ["Deret Asli", "Log", f"Box-Cox (λ={lam_opt:.2f})"],
    "Varians Total": [train.var(), train_log.var(), np.var(train_bc)],
    "Rasio Varians Max/Min Semester": [var_ratio(train.values), var_ratio(train_log.values), var_ratio(train_bc)]
})
print("Perbandingan Stabilitas Varians Antar-Semester:")
display(df_comp.round(3))


## 3. Pembahasan Temuan & Keputusan Transformasi

Dari pengujian di atas, terdapat dua pertimbangan yang saling melengkapi:

1. **Secara Statistik Matematis:**
   * Nilai $\hat{\lambda} = 0,1387$.
   * Uji rasio likelihood menolak $\lambda = 1$ ($p = 0,0019$) dan menerima $\lambda = 0$ ($p = 0,6108$).
   * Artinya, jika dilihat murni dari kriteria normalitas Box-Cox, transformasi logaritma didukung secara statistik.

2. **Secara Karakteristik Data & Metodologi:**
   * **Sumber heteroskedastisitas:** Kenaikan varians pada semester tertentu bertepatan dengan musim kemarau tahunan (pola musiman), bukan tren kenaikan varians eksponensial jangka panjang.
   * **Efektivitas perataan:** Rasio varians max/min semester pada deret asli ($3,46$) hanya turun sedikit ke $3,23$ pada deret log (penurunan $\approx 6,8\%$).
   * **Dekomposisi Kunto (Tahap 1):** STL aditif terbukti dominan ($F_s = 0,927$). Memaksakan log pada deret dengan musiman aditif dapat mengubah sifat musiman menjadi non-linear.
   * **Bias retransformasi:** Memodelkan log memerlukan koreksi bias (Jensen's inequality) saat inverse-transform agar ramalan rata-rata tidak *underestimate*. Satuan fisik asli (µg/m³) juga lebih mudah diinterpretasikan terhadap baku mutu lingkungan.

**Keputusan:**  
Sebagai basis utama pemodelan kelompok, **deret asli dipertahankan**. Namun, untuk menjaga objektivitas dan kredibilitas analisis, **uji stasioneritas pada tahap berikutnya dijalankan untuk kedua skala (deret asli dan deret log)** agar opsi keduanya terdokumentasi lengkap.


## 4. Uji Stasioneritas Awal Deret Level (ADF & KPSS)
Uji stasioneritas awal dilakukan menggunakan dua uji komplementer:
* **ADF (Augmented Dickey-Fuller):**  
  $H_0$: Deret memiliki akar unit / tidak stasioner.  
  $H_1$: Deret stasioner.  
  (Tolak $H_0$ jika $p < 0,05$).
* **KPSS (Kwiatkowski-Phillips-Schmidt-Shin):**  
  $H_0$: Deret stasioner di sekitar konstan.  
  $H_1$: Deret memiliki akar unit / tidak stasioner.  
  (Tolak $H_0$ jika $p < 0,05$).

Pengujian dilakukan pada data latih ($n=104$) dan data penuh ($n=116$), baik untuk deret asli maupun deret log.


In [ ]:
series_log = np.log(series)

def audit_stationarity(ts_data, label):
    adf = adfuller(ts_data, autolag="AIC")
    kp = kpss(ts_data, regression="c", nlags="auto")
    return [
        {
            "Skala": label,
            "Uji": "ADF",
            "Statistik": round(adf[0], 4),
            "p-value": round(adf[1], 4),
            "Kritis 1%": round(adf[4]["1%"], 3),
            "Kritis 5%": round(adf[4]["5%"], 3),
            "Lag": int(adf[2]),
            "n": int(adf[3]),
            "Kesimpulan": "Stasioner (Tolak H0)" if adf[1] < 0.05 else "Tidak stasioner (Ada unit root)"
        },
        {
            "Skala": label,
            "Uji": "KPSS",
            "Statistik": round(kp[0], 4),
            "p-value": round(kp[1], 4),
            "Kritis 1%": round(kp[3]["1%"], 3),
            "Kritis 5%": round(kp[3]["5%"], 3),
            "Lag": int(kp[2]),
            "n": len(ts_data),
            "Kesimpulan": "Tidak stasioner (Tolak H0)" if kp[1] < 0.05 else "Stasioner (Gagal tolak H0)"
        }
    ]

results = []
results.extend(audit_stationarity(train, "Deret Asli - Data Latih (n=104)"))
results.extend(audit_stationarity(series, "Deret Asli - Data Penuh (n=116)"))
results.extend(audit_stationarity(train_log, "Deret Log - Data Latih (n=104)"))
results.extend(audit_stationarity(series_log, "Deret Log - Data Penuh (n=116)"))

df_stat_all = pd.DataFrame(results)
display(df_stat_all)


## 5. Ringkasan & Rekomendasi untuk Tahap 2B (Aldo)

### Evaluasi Stasioneritas:
1. **Deret Asli:**
   * Pada data penuh ($n=116$), ADF menghasilkan $p = 0,2034 > 0,05$ (tidak stasioner, ada akar unit).
   * Pada data latih ($n=104$), ADF menghasilkan $p = 0,0235$, namun ACF data mentah pada Tahap 1 meluruh secara lambat (*slow decay*), mengindikasikan persistensi tinggi akibat fluktuasi musiman tahunan.
   * Uji KPSS pada kedua set gagal menolak $H_0$ ($p \ge 0,10$), yang umum terjadi pada sampel berukuran sedang dengan pergeseran bertahap (*low power*).
2. **Deret Log:**
   * Menunjukkan pola yang serupa pada data latih ($p = 0,0221$) dan stasioner pada data penuh ($p = 0,0222$).

---

### Catatan untuk Aldo (Tahap 2B):
1. **Diferensiasi Non-Musiman ($d=1$):** Deret asli tingkat level belum stasioner dalam rataan, sehingga diperlukan diferensiasi pertama:
   $$\Delta y_t = y_t - y_{t-1}$$
   *(Jika kelompok memilih pemodelan log, maka $\Delta \ln(y_t)$ yang digunakan).*
2. **Diferensiasi Musiman ($D$):** Karena panjang deret latih hanya mencakup 2 siklus musiman ($s=52$), penerapan $D=1$ berisiko memicu *over-differencing* (menghilangkan 52 amatan awal). Disarankan memprioritaskan $D=0$ dan menangkap struktur musiman melalui autoregresif musiman SARIMA.
